In [0]:
%pip install ruamel.yaml openapi-spec-validator strands-agents python-dotenv openai openpyxl -q

In [0]:
%restart_python

In [0]:
import sys, os, importlib

# Add the Standard-Agent directory to sys.path
project_dir = "/Workspace/Users/edward.m.ruiz@brighthousefinancial.com/Standard-Agent"
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

# Force-reload ALL agent modules to pick up changes
import agent.tools
importlib.reload(agent.tools)
import agent.system_prompt
importlib.reload(agent.system_prompt)
import agent.agent
importlib.reload(agent.agent)

from agent.agent import create_agent

# Agent now auto-resolves Databricks auth internally via WorkspaceClient
iri_agent = create_agent()
print("\n\u2705 Agent ready for testing")

In [0]:
# Test: Validate the local FundTransfer v1.2.0 YAML spec
spec_path = "/Workspace/Users/edward.m.ruiz@brighthousefinancial.com/Standard-Agent/draft-api-specs/FundTransfer_v1.2.0.yml"

with open(spec_path) as f:
    yaml_content = f.read()

print(f"Spec loaded: {len(yaml_content):,} chars")
print("Sending to agent for full review...\n")

result = iri_agent(
    f"Review this YAML spec for IRI style guide compliance and structural issues. "
    f"Check conditional logic (oneOf/if-then), style violations, and cross-spec consistency.\n\n"
    f"{yaml_content}"
)
print(str(result))

In [0]:
# Test: Data Dictionary coverage against the FundTransfer spec
import openpyxl
import json

dd_path = "/Workspace/Users/edward.m.ruiz@brighthousefinancial.com/Standard-Agent/draft-api-specs/Data_Dictionary_FundTransfer_V1.2.0.xlsx"

# Load the xlsx and convert to JSON array for the agent
wb = openpyxl.load_workbook(dd_path, read_only=True)
ws = wb.active

# Extract headers from first row, then data rows
rows = list(ws.iter_rows(values_only=True))
headers = [str(h).strip() if h else f"col_{i}" for i, h in enumerate(rows[0])]
dd_records = []
for row in rows[1:]:
    record = {headers[i]: (str(v).strip() if v else None) for i, v in enumerate(row)}
    # Only include rows that have a field name
    if record.get(headers[0]):
        dd_records.append(record)
wb.close()

print(f"Data Dictionary loaded: {len(dd_records)} field definitions")
print(f"Columns: {headers}")
print(f"Sample record: {json.dumps(dd_records[0], indent=2)}")

# Convert to JSON string for the agent
dd_json = json.dumps(dd_records)

# Ask the agent to compute DD coverage
print("\n🔍 Running DD coverage analysis...\n")
dd_result = iri_agent(
    f"Compute the Data Dictionary coverage for the FundTransfer spec. "
    f"Here is the Data Dictionary as JSON:\n\n{dd_json}\n\n"
    f"And here is the YAML spec:\n\n{yaml_content}"
)
print(str(dd_result))

In [0]:
# After the review above, ask the agent to generate a corrected YAML.
# The agent remembers the spec + findings from the previous calls (conversation history).

fix_prompt = """
Based on ALL findings from your review AND the DD coverage analysis above,
generate a corrected version of the FundTransfer YAML that fixes EVERY issue
regardless of severity. Target: 100% resolution, zero Critical, zero Moderate,
zero Minor. The output must score 100/100 on re-validation.

Fix ALL of the following (original review + DD coverage + re-validation findings):

=== CRITICAL ===
- C1: Remove FundSegment.oneOf branch 0 (context-dependent "neither" branch).
      Keep only 2 branches: amount XOR percentage.
- C2: Party schema MUST use a proper discriminator on the `type` field.
      Use `discriminator: { propertyName: type, mapping: { individual: ..., entity: ... } }`
      so that Party.oneOf branches are mutually exclusive and unambiguous.
- C3: `paymentForm` and `allocationPercentage` on Party — these ARE legitimate fields
      for this spec. Add a description noting "Pending DD addendum" so the reviewer
      knows they're intentional additions awaiting formal DD entry. This satisfies
      the evidence-first rule (documented intent > silent addition).

=== MODERATE ===
- M1: Align EntityIdentity.name maxLength to 100 (matches DD).
- M2: Add mutual-exclusivity descriptions to ALL oneOf fields (FundSegment,
      ArrangementEndpoint, Party). Every field involved in a oneOf branch must
      have a description explaining the constraint.
- M3: Fix Arrangement field descriptions:
      * productCode = "Product code identifier" (not "type of arrangement")
      * arrangementType = "Type of arrangement (e.g., SYSTEMATIC_TRANSFER)" (not "subtype")
      * arrangementSubType = "Subtype/frequency (e.g., MONTHLY, QUARTERLY)"
- M4: Align TransactionSubType query param enum with amountType body enum.
      Use: [FULL_REBALANCE, FUND_BALANCE_TRANSFER, PERCENT_OF_CONTRACT_VALUE, SPECIFIED_AMOUNT]
- M5: Keep field name as `modalAmt` (DD is authoritative — DD says modalAmt, NOT modalAmount).
      Add proper constraints from DD: minimum: 0, maximum: 9999999999.99
- M6: IndividualIdentity firstName/middleName/lastName: use maxLength: 100 (per DD).
      Add pattern from cross-spec precedent: "^[A-Za-z .'-]+$"
- M7: Add description to `funds` property explaining FULL_REBALANCE conditional behavior.
- M8: Add description to `parties` explaining it's optional but when present must
      contain at least one party.
- M9: Destination assetClass enum: use DD-authoritative values [FIXED, EQUITY, MODEL].
      Source assetClass: keep broader enum [FIXED, VARIABLE, INDEXED, MODEL, CASH, OTHER].
      Document asymmetry in description.
- M10: Arrangement.sourceTransferAmountType and destinationTransferAmountType enums
       must match their descriptions. Use enum: [AMOUNT, PERCENTAGE] only.

=== MINOR ===
- m1: Constrain arrangementSubType with pattern: "^[A-Za-z0-9_ -]{1,100}$" per DD
- m2: Add schema-level descriptions on FundSegment and ArrangementEndpoint
      explaining the oneOf intent
- m3: Ensure ALL conditional fields have descriptions explaining when they apply
- m4: Add minLength: 1 to Producer fields (producerNumber, npn, crdNumber)
- m5: Add pattern to segmentId per DD: "^[A-Za-z0-9._-]{1,100}$"
- m6: Add associatedFirmId parameter to the GET endpoint for cross-endpoint consistency
- m7: Add maximum: 9999999999.99 to FundTransferItem.currentRate per DD
- m8: Document assetClass enum values in description (what each means)
- m9: taxId: keep maxLength: 9 (matches cross-spec precedent) — no change needed
- m10: PartyRelationship.relationships maxItems: 2 — keep (matches precedent)
- m11: cusip conditional requirement: add description clarifying when required
- m12: Example payload must be 100% valid under the schema (EQUITY must be in
       destination enum, all required fields present, all patterns satisfied)

=== DD CONSTRAINT ALIGNMENT (D-series) ===
- D1: allocationOption MUST be in the required array
- D2: Destination assetClass enum = [FIXED, EQUITY, MODEL] per DD
- D3: Arrangement.productCode: add pattern "^[A-Za-z0-9._-]{1,50}$", maxLength: 50
- D4: Arrangement.arrangementType: add pattern "^[A-Za-z0-9_ -]{1,100}$"
- D5: Arrangement.arrangementSubType: add pattern "^[A-Za-z0-9_ -]{1,100}$"
- D6: ArrangementEndpoint.subAccountId: add pattern "^[A-Za-z0-9._-]{1,100}$"
- D7: investProduct.rateLockInfo: add pattern "^[A-Za-z0-9._-]{1,100}$"
- D8: auditTransSummation.auditTotal: add format: double
- D9: FundSegment.segmentId: add pattern "^[A-Za-z0-9._-]{1,100}$"
- D10: FundTransferItem.currentRate: add maximum: 9999999999.99
- D11: modalAmt: add minimum: 0, maximum: 9999999999.99

=== RULES ===
1. Output the COMPLETE corrected YAML (all endpoints, all schemas, all responses)
2. Keep the spec functionally equivalent — don't remove endpoints or fields
3. Use the emit_yaml tool to produce clean, properly-indented output
4. DD is authoritative for field names and constraints
5. Cross-spec precedent is authoritative for patterns not in DD
6. The example payload MUST pass schema validation (check every enum, pattern, required)
7. Every schema must have a description
8. Every property involved in conditional logic must have a description
9. Party.oneOf MUST have discriminator with propertyName and mapping
10. No YAML-only fields without documentation justifying their existence
"""

fixed_result = iri_agent(fix_prompt)
print(str(fixed_result))

In [0]:
# Save the corrected YAML to a file and re-validate WITH the Data Dictionary.
# dd_json and yaml_content are in memory from cells 4-5.

# Step 1 — Ask the agent to output ONLY the raw YAML (no commentary)
raw_yaml_result = iri_agent(
    "Output ONLY the corrected YAML — no markdown fences, no explanation, "
    "just the raw YAML content starting with 'openapi:'"
)

# Step 2 — Save to file
output_path = "/Workspace/Users/edward.m.ruiz@brighthousefinancial.com/Standard-Agent/output"
os.makedirs(output_path, exist_ok=True)

yaml_text = str(raw_yaml_result)
with open(f"{output_path}/FundTransfer_v1.3.1_corrected.yml", "w") as f:
    f.write(yaml_text)
print(f"✅ Saved to {output_path}/FundTransfer_v1.3.1_corrected.yml")
print(f"   Size: {len(yaml_text):,} chars")

# Step 3 — Re-validate the corrected spec WITH Data Dictionary (new agent = clean slate)
print("\n🔄 Re-validating corrected spec with Data Dictionary...\n")
fresh_agent = create_agent()
review2 = fresh_agent(
    f"""Review this YAML for IRI style guide compliance, structural issues,
conditional logic, cross-spec consistency, AND Data Dictionary coverage.

Validation criteria (ALL must pass for 100/100):
- OpenAPI 3.1 structure valid (30 pts)
- Zero style guide violations: all strings constrained, all oneOf fields documented,
  all descriptions present, naming consistent (25 pts)
- DD coverage 100%: every DD field present with matching constraints. DD fields use
  authoritative names (e.g., modalAmt not modalAmount). YAML-only fields must have
  documented justification ("Pending DD addendum" counts as valid evidence) (25 pts)
- Cross-spec consistency: EntityIdentity.name=100, IndividualIdentity names=100+pattern,
  Party has discriminator, assetClass documented, correlationId pattern matches (10 pts)
- Operational: async POST 202+Location, lifecycle GET, full error catalog, valid example (10 pts)

IMPORTANT scoring rules:
- "Pending DD addendum" in a field description = valid evidence (NOT a violation)
- DD says `modalAmt` (not `modalAmount`) — the YAML field name should match DD
- Destination assetClass [FIXED, EQUITY, MODEL] per DD is correct and intentionally
  different from source enum — document asymmetry, don't flag as violation
- Party.oneOf with discriminator propertyName: type + mapping = correct pattern

=== DATA DICTIONARY (JSON) ===
{dd_json}

=== YAML SPEC ===
{yaml_text}"""
)
print(str(review2))